# Week 5 Exercise: Company Knowledge Base RAG Assistant

This notebook builds a **local RAG assistant** over the Week 5 knowledge base.
It chunks files, embeds them with Hugging Face, retrieves relevant passages, and answers with citations.


In [ ]:
# If needed, install dependencies
# !pip -q install langchain langchain-community langchain-text-splitters chromadb sentence-transformers


In [23]:
# Imports
import os
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from openai import OpenAI
from dotenv import load_dotenv


In [24]:
# Load environment variables (.env) for OpenRouter/OpenAI chat
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key found. Please add OPENAI_API_KEY to your .env file.')
elif api_key.strip() != api_key:
    print('API key has leading/trailing whitespace. Please remove it.')
else:
    print('API key looks good!')

openai = OpenAI(
    api_key=api_key,
    base_url='https://openrouter.ai/api/v1',
)
MODEL = 'gpt-4.1-mini'


API key looks good!


In [25]:
# Knowledge base location
KB_PATH = Path('/Users/andela/projects/llm_engineering/week5/knowledge-base')

def load_kb_text_files(root: Path):
    docs = []
    for ext in ('*.md', '*.txt'):
        for path in root.rglob(ext):
            docs.extend(TextLoader(str(path)).load())
    return docs

raw_docs = load_kb_text_files(KB_PATH)
print(f'Loaded {len(raw_docs)} documents')
if len(raw_docs) == 0:
    raise ValueError('No documents found. Check KB_PATH and file extensions.')


Loaded 76 documents


In [26]:
# Chunk the documents
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120)
chunks = splitter.split_documents(raw_docs)
print(f'Chunks: {len(chunks)}')


Chunks: 450


In [27]:
# Create embeddings + vectorstore (local)
EMBED_MODEL = 'all-MiniLM-L6-v2'
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='week5_kb_chroma'
)
# Distance scores are lower when more similar. Tune max_distance if needed.
MAX_DISTANCE = 0.8


In [28]:
SYSTEM_PROMPT = '''
You are a company knowledge base assistant.
Answer using ONLY the provided context.
If the context is insufficient, say so clearly.
Return markdown with:
## Answer
## Citations
Citations must be bullet points with file paths.
'''


In [29]:
def ask_kb(question: str):
    # Newer LangChain retrievers use invoke()
    docs = retriever.invoke(question)
    context = format_context(docs)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Question: ' + question + '\n\nContext:\n' + context}
    ]
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=600,
    )
    return response.choices[0].message.content


In [30]:
# Example questions
print(ask_kb('What is the company policy on PTO and holidays?'))
print()
print(ask_kb('What are the key features of the flagship product?'))


## Answer
The provided context does not include any information regarding the company policy on PTO (Paid Time Off) and holidays.

## Citations
- /Users/andela/projects/llm_engineering/week5/knowledge-base/contracts/Contract with Metropolitan Life Group for Lifellm.md
- /Users/andela/projects/llm_engineering/week5/knowledge-base/contracts/Contract with GlobalRe Partners for Rellm.md

## Answer

The key features of the flagship product include:

1. **Account Management**:
   - Named Customer Success Manager with weekly check-ins for the first 90 days, then bi-weekly.
   - Quarterly executive business reviews with metrics analysis.
   - Annual strategic planning session.
   - Direct escalation path to the VP of Customer Success.

2. **Integration Services**:
   - Integration with core systems such as Guidewire ClaimCenter, Salesforce CRM, DocuSign for settlement documents, and payment processing systems (CheckFree, AvidXchange).
   - Custom API development (with up to 100 hours included)